In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm  
import nltk
#nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
import json
from pathlib import Path
import re
import wrds

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
base = Path('/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/transcripts')

def load_transcript_text(filepath):
    with open(filepath) as f:
        data = json.load(f)
    if data and isinstance(data, list) and len(data) > 0:
        return data[0].get('content', '')
    return ''

manifest = []
for filepath in base.rglob('*.json'):
    symbol = filepath.parent.name
    stem = filepath.stem
    match = re.match(r'(\d{4})Q(\d)', stem)
    if match:
        year, quarter = int(match.group(1)), int(match.group(2))
        manifest.append({
            'symbol': symbol,
            'year': year,
            'quarter': quarter,
            'filepath': filepath,
        })
manifest_df = pd.DataFrame(manifest)

In [ ]:
transcript_df = manifest_df.copy()
transcript_df['transcript'] = transcript_df['filepath'].apply(load_transcript_text)

In [ ]:
DATA_RAW = Path('/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw')

# Step 1: Pull crsp.msp500list if you haven't already
sp500_path = DATA_RAW / 'crsp_msp500list.parquet'
if not sp500_path.exists():
    print("Pulling crsp.msp500list from WRDS...")
    db = wrds.Connection()
    sp500 = db.raw_sql("SELECT permno, start, ending FROM crsp.msp500list")
    sp500.to_parquet(sp500_path)
    db.close()
    print(f"Saved {len(sp500)} rows to {sp500_path}")
else:
    sp500 = pd.read_parquet(sp500_path)
    print(f"Loaded existing S&P 500 list: {len(sp500)} rows")

# Step 2: Get set of PERMNOs that were ever in the S&P 500
sp500_permnos = set(sp500['permno'].unique())
print(f"Unique historical S&P 500 PERMNOs: {len(sp500_permnos):,}")

# Step 3: Map PERMNOs to tickers via CRSP
crsp_dsf = pd.read_parquet(DATA_RAW / 'crsp_dsf.parquet')
sp500_tickers = set(
    crsp_dsf[crsp_dsf['permno'].isin(sp500_permnos)]['ticker']
    .dropna()
    .str.upper()
    .unique()
)
print(f"Unique S&P 500 historical tickers: {len(sp500_tickers):,}")

# Step 4: Filter transcript_df
before = len(transcript_df)
transcript_df_sp500 = transcript_df[
    transcript_df['symbol'].str.upper().isin(sp500_tickers)
].copy()
print(f"Filtered: {before:,} → {len(transcript_df_sp500):,} transcripts")

In [ ]:
before = len(transcript_df_sp500)
transcript_df_sp500 = transcript_df_sp500[
    transcript_df_sp500['transcript'].apply(
        lambda x: isinstance(x, str) and len(x.strip()) > 0
    )
].copy()
print(f"Dropped {before - len(transcript_df_sp500):,} rows with missing/empty content")
print(f"Remaining: {len(transcript_df_sp500):,} transcripts")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
model.eval()

In [ ]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = model.to(device)

OUTPUT_DIR = Path('../data/interim/finbert_sentences')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def score_transcript(row, batch_size=64):
    """Score one transcript with FinBERT. Saves to parquet. Idempotent."""
    output_path = OUTPUT_DIR / row['symbol'] / f"{row['year']}Q{row['quarter']}.parquet"
    
    if output_path.exists():
        return 'skipped'
        
    transcript = row['transcript']
    if not isinstance(transcript, str) or not transcript.strip():
        return 'empty'
        
    sentences = sent_tokenize(row['transcript'])
    if not sentences:
        return 'empty'
    
    all_probs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, 
                           return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
    
    probs_array = np.vstack(all_probs)
    
    result = pd.DataFrame({
        'sentence_idx': range(len(sentences)),
        'sentence': sentences,
        'finbert_pos': probs_array[:, 0],
        'finbert_neg': probs_array[:, 1],
        'finbert_neu': probs_array[:, 2],
    })
    
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_parquet(output_path)
    return 'saved'

stats = {'saved': 0, 'skipped': 0, 'empty': 0}
for i, (_, row) in enumerate(tqdm(transcript_df_sp500.iterrows(), total=len(transcript_df_sp500))):
    result = score_transcript(row)
    stats[result] += 1
    
    if (i + 1) % 100 == 0 and device == 'mps':
        torch.mps.empty_cache()

print(stats)

In [ ]:
INTERIM_DIR = Path('../data/interim/finbert_sentences')

records = []
for filepath in tqdm(list(INTERIM_DIR.rglob('*.parquet'))):
    symbol = filepath.parent.name
    stem = filepath.stem
    year = int(stem[:4])
    quarter = int(stem[5])
    
    df = pd.read_parquet(filepath)
    if len(df) == 0:
        continue
    
    records.append({
        'symbol': symbol,
        'year': year,
        'quarter': quarter,
        'n_sentences': len(df),
        'mean_pos': df['finbert_pos'].mean(),
        'mean_neg': df['finbert_neg'].mean(),
        'mean_neu': df['finbert_neu'].mean(),
        'net_tone': df['finbert_pos'].mean() - df['finbert_neg'].mean(),
        'share_strong_pos': (df['finbert_pos'] > 0.8).mean(),
        'share_strong_neg': (df['finbert_neg'] > 0.8).mean(),
    })

finbert_scores = pd.DataFrame(records)
finbert_scores.to_parquet('../data/interim/finbert_scores.parquet')
print(f"Built FinBERT scores: {len(finbert_scores)} calls")
print(finbert_scores.head())
print(f"\nNet tone distribution:")
print(finbert_scores['net_tone'].describe())

In [ ]:
# Distribution check
print("Distribution of net_tone:")
print(finbert_scores['net_tone'].describe())

# Reasonable: roughly bell-shaped, centered slightly above zero (calls are slightly more positive than negative on average), range probably [-0.4, +0.6]
# Red flags: bimodal, way off-center, all very close to zero, or extreme outliers

# Sample some calls and check
top_positive = finbert_scores.nlargest(5, 'net_tone')[['symbol', 'year', 'quarter', 'net_tone']]
top_negative = finbert_scores.nsmallest(5, 'net_tone')[['symbol', 'year', 'quarter', 'net_tone']]
print("\nMost positive calls:")
print(top_positive)
print("\nMost negative calls:")
print(top_negative)

# Coverage check  
print(f"\nUnique firms: {finbert_scores['symbol'].nunique()}")
print(f"Calls per year:")
print(finbert_scores.groupby('year').size())

In [ ]:
print("NaN counts:")
print(finbert_scores[['mean_pos', 'mean_neg', 'mean_neu', 'net_tone', 
                       'share_strong_pos', 'share_strong_neg']].isna().sum())

# Also check sentence counts — flag anything suspicious
print(f"\nSentence count distribution:")
print(finbert_scores['n_sentences'].describe())
print(f"\nMin sentence calls (potential parsing issues):")
print(finbert_scores.nsmallest(5, 'n_sentences')[['symbol', 'year', 'quarter', 'n_sentences']])